**Configuration note:** this notebook evaluates annotated paralog loci by BUSCO-calibrated HiFi read-depth. It consumes the read-depth-screen output (locus classification + pair integration) and the paralog-family genomic summary. Input paths mirror `config.sh` at the repo root.


In [ ]:
# Input paths — edit for your environment (mirror config.sh at the repo root).
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


# BUSCO-calibrated HiFi read-depth evaluation of annotated paralog loci

**Purpose**: This notebook evaluates HiFi read depth first at the level of individual
paralog loci (Stage 1), then integrates locus-level observations into pair-level
biological interpretations (Stage 2).

**Data sources**:
- `read-depth-screen-v4-busco-calibrated/locus_classification_v4_busco.tsv` — 3,347 loci
- `read-depth-screen-v4-busco-calibrated/pair_integration_v4_busco.tsv` — 2,170 pairs
- `paralog_genomic_summary.tsv` — paralog metadata (type, expression, genomic context)

**V4 thresholds** (BUSCO-calibrated from 12,057 complete autosomal single-copy loci):
- Expected depth: [0.636, 1.394] — central 95% of autosomal single-copy BUSCO
- Half-depth-like: [0.30, 0.55] — false half-depth rate 0.6% among single-copy BUSCO
- Very-low: [0.20, 0.30) — below empirical single-copy BUSCO range (min = 0.394)
- Intermediate: (0.55, 0.636)


## 1. Load and prepare data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')
import re
import textwrap

# ---------------------------------------------------------------------------
# Publication-style settings
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'font.size': 9,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'legend.fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# ---------------------------------------------------------------------------
# Color palettes
# ---------------------------------------------------------------------------
LOCUS_COLORS = {
    'normal_depth': '#2ECC40',
    'consistent_half_depth': '#E74C3C',
    'very_low_coverage': '#FF4136',
    'localized_or_asymmetric_low_depth': '#FF851B',
    'unique_mapping_deficit': '#9B59B6',
    'inconclusive': '#AAAAAA',
}

PAIR_COLORS = {
    'both_expected_depth': '#2ECC40',
    'one_expected_one_atypical': '#FFDC00',
    'both_atypical': '#F39C12',
    'asymmetric_coverage': '#FF851B',
    'unresolved': '#AAAAAA',
    'concordant_half_depth_sum_valid': '#E74C3C',
}

PARALOG_TYPE_COLORS = {
    'l': '#3498DB',
    'dl': '#E67E22',
    'rl': '#2ECC40',
}

# ---------------------------------------------------------------------------
# Display-name mappings (publication-facing labels)
# Internal names are preserved in code and TSV outputs; these are used
# only for figures, tables, legends, and panel annotations.
# ---------------------------------------------------------------------------
LOCUS_DISPLAY = {
    'normal_depth': 'EXPECTED DEPTH',
    'consistent_half_depth': 'HALF-DEPTH',
    'very_low_coverage': 'VERY LOW COVERAGE',
    'localized_or_asymmetric_low_depth': 'LOCALIZED LOW DEPTH',
    'unique_mapping_deficit': 'MAPPING AMBIGUITY',
    'inconclusive': 'UNEVALUABLE',
}

PAIR_DISPLAY = {
    'concordant_half_depth_sum_valid': 'CONCORDANT HALF-DEPTH',
    'both_expected_depth': 'BOTH LOCI AT EXPECTED DEPTH',
    'asymmetric_coverage': 'ASYMMETRIC COVERAGE',
    'one_expected_one_atypical': 'ONE EXPECTED, ONE ATYPICAL',
    'both_atypical': 'BOTH ATYPICAL',
    'unresolved': 'INDETERMINATE',
}

LOCUS_INTERPRETATION = {
    'normal_depth': (
        'Coverage within the empirical\n'
        'range of single-copy BUSCO\n'
        'genes. Consistent with the\n'
        'expected genomic representation\n'
        'of the locus.'
    ),
    'consistent_half_depth': (
        'Persistent locus-wide half-\n'
        'depth signal. Overlaps extreme\n'
        'single-copy BUSCO tail (false half-depth\n'
        'rate 0.6%). Requires pair-level\n'
        'and genomic-context\n'
        'interpretation.'
    ),
    'very_low_coverage': (
        'Below the empirical range of\n'
        'single-copy autosomal BUSCO\n'
        'genes (min=0.394). More\n'
        'consistent with mapping\n'
        'dropout, deletions, or assembly\n'
        'collapse than with haploid\n'
        'copy number.'
    ),
    'localized_or_asymmetric_low_depth': (
        'Localized coverage anomaly;\n'
        'review for mapping, CNV,\n'
        'repeat, or assembly-boundary\n'
        'effects.'
    ),
    'unique_mapping_deficit': (
        'Coverage is supported, but\n'
        'locus-specific read assignment\n'
        'is limited by high sequence\n'
        'similarity.'
    ),
    'inconclusive': (
        'Coverage cannot be interpreted\n'
        'confidently; manual review may\n'
        'be required.'
    ),
}

PAIR_INTERPRETATION_TEXT = {
    'concordant_half_depth_sum_valid': (
        'Both loci exhibit concordant\n'
        'half-depth coverage with a\n'
        'valid sum check. Pattern is\n'
        'consistent with a retained\n'
        'haplotypic duplication.'
    ),
    'both_expected_depth': (
        'Both loci exhibit expected\n'
        'diploid coverage. Pattern is\n'
        'consistent with independently\n'
        'assembled paralogous loci.'
    ),
    'asymmetric_coverage': (
        'The two loci exhibit different\n'
        'coverage states. Pattern may\n'
        'reflect copy-number differences,\n'
        'assembly effects, or biological\n'
        'divergence.'
    ),
    'one_expected_one_atypical': (
        'One locus exhibits expected\n'
        'coverage while the other shows\n'
        'an atypical coverage pattern.\n'
        'Additional genomic evidence is\n'
        'required.'
    ),
    'both_atypical': (
        'Coverage signals are present but\n'
        'form no clean pair-level pattern.\n'
        'Pattern may reflect genuine\n'
        'paralogs in complex genomic regions.\n'
        'External evidence required.'
    ),
    'unresolved': (
        'Neither locus provides evaluable\n'
        'coverage data (both flanks\n'
        'uninformative or no pattern\n'
        'matched). Coverage cannot\n'
        'contribute to interpretation.'
    ),
}

# ---------------------------------------------------------------------------
# BUSCO-calibrated thresholds
# ---------------------------------------------------------------------------
EXPECTED_LO, EXPECTED_HI = 0.636, 1.394
HALF_LO, HALF_HI = 0.30, 0.55
VERY_LOW_LO, VERY_LOW_HI = 0.20, 0.30

# Locus category display order
LOCUS_ORDER = ['normal_depth', 'consistent_half_depth', 'very_low_coverage',
               'localized_or_asymmetric_low_depth', 'unique_mapping_deficit',
               'inconclusive']

# Pair interpretation order (by count, most → least)
PAIR_ORDER = ['both_expected_depth', 'one_expected_one_atypical',
              'both_atypical', 'asymmetric_coverage',
              'unresolved', 'concordant_half_depth_sum_valid']

PARALOG_TYPE_ORDER = ['l', 'dl', 'rl']


In [ ]:
# Load v4 BUSCO-calibrated locus classification
loci_v4 = pd.read_csv(
    f'{PROJ_ROOT}/figure/paralog-alignment-visualization/read-depth-screen-v4-busco-calibrated/locus_classification_v4_busco.tsv', sep='\t'
)
print(f'=== V4 BUSCO-calibrated Locus classification ===')
print(f'Total loci: {len(loci_v4)}  (parents: {(loci_v4["gene_type"] == "parent").sum()}, '
      f'paralogs: {(loci_v4["gene_type"] != "parent").sum()})')

# Separate paralogs and parents
paralog_loci = loci_v4[loci_v4['gene_type'] != 'parent'].copy()
parent_loci = loci_v4[loci_v4['gene_type'] == 'parent'].copy()

print(f'\nLocus coverage categories (paralogs only, n={len(paralog_loci)}):')
for cat in LOCUS_ORDER:
    cnt = (paralog_loci['locus_coverage_category'] == cat).sum()
    if cnt > 0:
        pct = 100 * cnt / len(paralog_loci)
        print(f'  {cat:45s} {cnt:5d}  ({pct:5.1f}%)')

print(f'\nThreshold version: {paralog_loci["threshold_version"].iloc[0]}')
print(f'BUSCO calibration N: {paralog_loci["busco_calibration_n"].iloc[0]}')
print(f'Expected depth: [{EXPECTED_LO}, {EXPECTED_HI}]')
print(f'Half-depth range: [{HALF_LO}, {HALF_HI}]')
print(f'Very-low range: [{VERY_LOW_LO}, {VERY_LOW_HI})')


In [ ]:
# Load v4 BUSCO-calibrated pair integration
pairs_v4 = pd.read_csv(
    f'{PROJ_ROOT}/figure/paralog-alignment-visualization/read-depth-screen-v4-busco-calibrated/pair_integration_v4_busco.tsv', sep='\t'
)
print(f'=== V4 BUSCO-calibrated Pair integration ===')
print(f'Total pairs: {len(pairs_v4)}')
for cat in PAIR_ORDER:
    cnt = (pairs_v4['pair_interpretation'] == cat).sum()
    if cnt > 0:
        pct = 100 * cnt / len(pairs_v4)
        print(f'  {cat:45s} {cnt:5d}  ({pct:5.1f}%)')


In [ ]:
# Load genomic summary for reference (paralog_type already in locus table)
genomic_summary = pd.read_csv(f'{PROJ_ROOT}/figure/paralog-alignment-visualization/paralog_genomic_summary.tsv', sep='\t')
print(f'Genomic summary: {len(genomic_summary)} paralogs')
print(f'Paralog types in locus table: {paralog_loci["paralog_type"].value_counts().to_dict()}')


## Panel A — Locus-level coverage classification criteria

Each of the 3,347 loci is classified independently by comparing its own
coverage (gene body + upstream/downstream flanks) to its chromosome-class
baseline (autosomal, chrX, or chrY).

**BUSCO-calibrated thresholds** (12,057 complete autosomal single-copy loci):

| Range | Interval | Rationale |
|-------|----------|-----------|
| Expected depth | [0.636, 1.394] | Central 95% of autosomal single-copy BUSCO |
| Half-depth-like | [0.30, 0.55] | ~C/2; false half-depth rate 0.6% among single-copy BUSCO |
| Very-low | [0.20, 0.30) | Below empirical single-copy BUSCO range (min = 0.394) |
| Intermediate | (0.55, 0.636) | Explicit gap between half-depth and expected |

**Decision logic** (from `calibrate_and_classify_v4.py`, first match wins):

1. Both flanks uninformative → `inconclusive`
2. Gene body unique < expected AND permissive > expected, both flanks normal → `unique_mapping_deficit`
3. Both flanks very-low AND combined very-low/half, both flanks informative → `very_low_coverage`
4. Both flanks half-depth AND combined half-depth, both flanks informative → `consistent_half_depth`
5. Both flanks expected AND combined expected, both flanks informative, gene body NOT half-depth → `normal_depth`
6. Any region at half-depth, very-low, or intermediate → `localized_or_asymmetric_low_depth`
7. Everything else → `inconclusive`


In [ ]:
# ============================================================================
# Panel A — Locus coverage classification criteria (ax.table)
# ============================================================================

locus_criteria_examples = [
    (
        'EXPECTED DEPTH',
        '#2ECC40',
        'Both flanks informative and\nboth within expected-depth\nrange [0.636–1.394]',
        'Unique gene-body depth must\nNOT fall in the half-depth\nrange [0.30–0.55]',
        'Combined gene ±25 kb depth\nmust be within expected-depth\nrange [0.636–1.394]',
        LOCUS_INTERPRETATION['normal_depth'],
    ),
    (
        'HALF-DEPTH',
        '#E74C3C',
        'Both flanks informative and\nboth within half-depth-like\nrange [0.30–0.55]',
        'Gene body is not required to\nbe half-depth; paralogous\nsequence may distort gene-body\nmapping',
        'Combined gene ±25 kb depth\nmust also fall within half-depth\nrange [0.30–0.55]',
        LOCUS_INTERPRETATION['consistent_half_depth'],
    ),
    (
        'VERY LOW COVERAGE',
        '#FF4136',
        'Both flanks informative and\nboth within very-low range\n[0.20–0.30)',
        'Gene body not required;\ncombined region must also\nbe very-low or half-depth',
        'Combined region also very-low\nor half-depth-like relative\nto chr-class baseline',
        LOCUS_INTERPRETATION['very_low_coverage'],
    ),
    (
        'LOCALIZED LOW DEPTH',
        '#FF851B',
        'At least one flank\ninformative; one-sided,\nasymmetric, intermediate,\nor conflicting flank pattern',
        'Half-depth [0.30–0.55],\nvery-low [0.20–0.30), or\nintermediate (0.55–0.636)\ngene-body depth may contribute',
        'Combined depth may be half-depth\n[0.30–0.55], very-low\n[0.20–0.30), intermediate\n(0.55–0.636), or inconsistent\nwith flanks',
        LOCUS_INTERPRETATION['localized_or_asymmetric_low_depth'],
    ),
    (
        'MAPPING AMBIGUITY',
        '#9B59B6',
        'Both flanks informative and\nnear expected depth\n[0.636–1.394]',
        'Unique gene-body depth < 0.636×C\nwhile permissive gene-body depth\n> 0.636×C — reads are present but\ncannot be uniquely assigned',
        'Combined-region depth does not\noverride the normal flank\nsupport',
        LOCUS_INTERPRETATION['unique_mapping_deficit'],
    ),
    (
        'UNEVALUABLE',
        '#AAAAAA',
        'Both flanks uninformative\n(≥50% zero-coverage) OR no\ndefined pattern is matched',
        'Missing, intermediate, or\ncontradictory evidence',
        'Missing, elevated (>1.394),\nintermediate (0.55–0.636), or\ncontradictory evidence',
        LOCUS_INTERPRETATION['inconclusive'],
    ),
]

col_labels = [
    'Category',
    'Flank requirements',
    'Gene-body role',
    'Combined-region role',
    'Interpretation',
]

# ============================================================================
# Build and draw locus criteria table with automatic wrapping
# ============================================================================

# Relative widths; must sum to 1.0
col_widths = [0.14, 0.18, 0.20, 0.20, 0.28]

# Approximate characters per wrapped line in each column.
wrap_widths = [22, 30, 34, 35, 43]

def normalize_text(text):
    return re.sub(r'\s+', ' ', str(text)).strip()


def wrap_text(text, width):
    return textwrap.fill(
        normalize_text(text),
        width=width,
        break_long_words=False,
        break_on_hyphens=False,
    )


# ---------------------------------------------------------------------------
# Print verification
# ---------------------------------------------------------------------------

print('=' * 100)
print('PANEL A — LOCUS CRITERIA TABLE')
print('=' * 100)

for i, (cat_name, cat_color, flank_req, gb_role, comb_role, interp) in enumerate(
        locus_criteria_examples, start=1):
    print(f'\n{"─" * 100}')
    print(f'{i}. {normalize_text(cat_name)}')
    print(f'   Flank requirements:   {normalize_text(flank_req)}')
    print(f'   Gene-body role:       {normalize_text(gb_role)}')
    print(f'   Combined-region role: {normalize_text(comb_role)}')
    print(f'   Interpretation:       {normalize_text(interp)}')


# ---------------------------------------------------------------------------
# Build wrapped table contents
# ---------------------------------------------------------------------------

table_data = []
table_colors_locus = []
row_line_counts = []

for cat_name, cat_color, flank_req, gb_role, comb_role, interp in locus_criteria_examples:

    raw_values = [cat_name, flank_req, gb_role, comb_role, interp]

    wrapped_values = [
        wrap_text(value, width)
        for value, width in zip(raw_values, wrap_widths)
    ]

    table_data.append(wrapped_values)

    table_colors_locus.append([
        cat_color, '#FAFAFA', '#FAFAFA', '#FAFAFA', '#FAFAFA',
    ])

    max_lines = max(v.count('\n') + 1 for v in wrapped_values)
    row_line_counts.append(max_lines)




In [ ]:
# ============================================================================
# Panel A — Publication-ready locus coverage criteria table
#
# Changes in this version:
#   1. All table fonts increased by +2 pt
#   2. Slightly more top/bottom cell padding for better readability
#   3. Manual cell drawing for precise control
#   4. Text wraps according to actual rendered width
#   5. Row height scales automatically to fit wrapped text cleanly
# ============================================================================

import re
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from matplotlib.font_manager import FontProperties


# ---------------------------------------------------------------------------
# Column labels
# ---------------------------------------------------------------------------

col_labels = [
    'Category',
    'Flank requirements',
    'Gene-body role',
    'Combined-region role',
    'Interpretation',
]


# ---------------------------------------------------------------------------
# Column widths
# Must sum to exactly 1.0
# ---------------------------------------------------------------------------

col_widths = [
    0.15,   # Category
    0.20,   # Flank requirements
    0.20,   # Gene-body role
    0.18,   # Combined-region role
    0.27,   # Interpretation
]

assert abs(sum(col_widths) - 1.0) < 1e-8


# ---------------------------------------------------------------------------
# Figure and typography settings
# ---------------------------------------------------------------------------

FIGURE_WIDTH_IN = 20.0

TITLE_FONTSIZE = 22
HEADER_FONTSIZE = 15
CATEGORY_FONTSIZE = 13.5
BODY_FONTSIZE = 12.5

# Slightly roomier line spacing for publication readability
BODY_LINE_SPACING = 1.02

# Internal cell padding, in points
HORIZONTAL_PADDING_PT = 6.5
VERTICAL_PADDING_PT = 5.0

# Header height, in points
HEADER_HEIGHT_PT = 48.0

# Figure margins, in points
TOP_MARGIN_PT = 6.0
TITLE_HEIGHT_PT = 28.0
TITLE_TABLE_GAP_PT = 8.0
BOTTOM_MARGIN_PT = 6.0

# Table position across the figure width
TABLE_LEFT = 0.02
TABLE_WIDTH = 0.96

# Cell appearance
BODY_BACKGROUND = '#FAFAFA'
HEADER_BACKGROUND = '#333333'
GRID_COLOR = '#666666'
GRID_LINEWIDTH = 0.65


# ---------------------------------------------------------------------------
# Font properties used for accurate text measurement
# ---------------------------------------------------------------------------

body_font = FontProperties(
    family='sans-serif',
    size=BODY_FONTSIZE,
    weight='normal',
)

category_font = FontProperties(
    family='sans-serif',
    size=CATEGORY_FONTSIZE,
    weight='bold',
)

header_font = FontProperties(
    family='sans-serif',
    size=HEADER_FONTSIZE,
    weight='bold',
)

title_font = FontProperties(
    family='sans-serif',
    size=TITLE_FONTSIZE,
    weight='bold',
)


# ---------------------------------------------------------------------------
# Text helpers
# ---------------------------------------------------------------------------

def normalize_text(text):
    """Remove manual line breaks and collapse repeated whitespace."""
    return re.sub(r'\s+', ' ', str(text)).strip()


def rendered_text_width(renderer, text, font_properties):
    """Measure text width in display pixels."""
    width, _, _ = renderer.get_text_width_height_descent(
        text,
        font_properties,
        ismath=False,
    )
    return width


def wrap_to_rendered_width(
    text,
    max_width_px,
    renderer,
    font_properties,
):
    """
    Wrap text using actual rendered width instead of character count.
    """

    text = normalize_text(text)

    if not text:
        return ''

    words = text.split()
    lines = []
    current_line = words[0]

    for word in words[1:]:

        candidate_line = f'{current_line} {word}'

        candidate_width = rendered_text_width(
            renderer,
            candidate_line,
            font_properties,
        )

        if candidate_width <= max_width_px:
            current_line = candidate_line
        else:
            lines.append(current_line)
            current_line = word

    lines.append(current_line)

    return '\n'.join(lines)


# ---------------------------------------------------------------------------
# Print verification
# ---------------------------------------------------------------------------

print('=' * 100)
print('PANEL A — LOCUS CRITERIA TABLE')
print('=' * 100)

for i, (
    cat_name,
    cat_color,
    flank_req,
    gb_role,
    comb_role,
    interp,
) in enumerate(locus_criteria_examples, start=1):

    print(f'\n{"─" * 100}')
    print(f'{i}. {normalize_text(cat_name)}')
    print(f'   Flank requirements:   {normalize_text(flank_req)}')
    print(f'   Gene-body role:       {normalize_text(gb_role)}')
    print(f'   Combined-region role: {normalize_text(comb_role)}')
    print(f'   Interpretation:       {normalize_text(interp)}')


# ---------------------------------------------------------------------------
# Create temporary figure/canvas for measuring text
# ---------------------------------------------------------------------------

fig = plt.figure(
    figsize=(FIGURE_WIDTH_IN, 8.0),
    facecolor='white',
)

ax_table = fig.add_axes([0, 0, 1, 1])
ax_table.set_xlim(0, 1)
ax_table.set_ylim(0, 1)
ax_table.axis('off')

fig.canvas.draw()
renderer = fig.canvas.get_renderer()

figure_width_px = FIGURE_WIDTH_IN * fig.dpi
table_width_px = figure_width_px * TABLE_WIDTH

horizontal_padding_px = (
    HORIZONTAL_PADDING_PT / 72.0
    * fig.dpi
)


# ---------------------------------------------------------------------------
# Wrap every cell using its actual physical column width
# ---------------------------------------------------------------------------

wrapped_rows = []
row_line_counts = []

for (
    cat_name,
    cat_color,
    flank_req,
    gb_role,
    comb_role,
    interp,
) in locus_criteria_examples:

    raw_values = [
        cat_name,
        flank_req,
        gb_role,
        comb_role,
        interp,
    ]

    wrapped_values = []

    for ci, value in enumerate(raw_values):

        cell_width_px = table_width_px * col_widths[ci]

        available_width_px = (
            cell_width_px
            - 2 * horizontal_padding_px
            - 4
        )

        selected_font = category_font if ci == 0 else body_font

        wrapped_value = wrap_to_rendered_width(
            value,
            available_width_px,
            renderer,
            selected_font,
        )

        wrapped_values.append(wrapped_value)

    wrapped_rows.append({
        'text': wrapped_values,
        'color': cat_color,
    })

    row_line_counts.append(
        max(
            value.count('\n') + 1
            for value in wrapped_values
        )
    )


# ---------------------------------------------------------------------------
# Calculate row heights
# ---------------------------------------------------------------------------

# A slightly larger multiplier gives cleaner publication spacing
body_line_height_pt = BODY_FONTSIZE * 1.08

row_heights_pt = []

for line_count in row_line_counts:

    row_height_pt = (
        line_count * body_line_height_pt
        + 2 * VERTICAL_PADDING_PT
    )

    row_heights_pt.append(row_height_pt)


table_height_pt = (
    HEADER_HEIGHT_PT
    + sum(row_heights_pt)
)

figure_height_pt = (
    TOP_MARGIN_PT
    + TITLE_HEIGHT_PT
    + TITLE_TABLE_GAP_PT
    + table_height_pt
    + BOTTOM_MARGIN_PT
)

figure_height_in = figure_height_pt / 72.0

# Resize figure to match content exactly
fig.set_size_inches(
    FIGURE_WIDTH_IN,
    figure_height_in,
    forward=True,
)

fig.canvas.draw()


# ---------------------------------------------------------------------------
# Convert points to axes fractions
# ---------------------------------------------------------------------------

def points_to_y_fraction(points):
    return points / figure_height_pt


title_y = 1.0 - points_to_y_fraction(TOP_MARGIN_PT)

table_top = 1.0 - points_to_y_fraction(
    TOP_MARGIN_PT
    + TITLE_HEIGHT_PT
    + TITLE_TABLE_GAP_PT
)

header_height = points_to_y_fraction(HEADER_HEIGHT_PT)

row_heights = [
    points_to_y_fraction(value)
    for value in row_heights_pt
]

horizontal_padding_fraction = (
    HORIZONTAL_PADDING_PT / 72.0
    / FIGURE_WIDTH_IN
)


# ---------------------------------------------------------------------------
# Draw title
# ---------------------------------------------------------------------------

ax_table.text(
    0.5,
    title_y,
    'Genomic read-depth criteria for paralog loci',
    fontproperties=title_font,
    ha='center',
    va='top',
    color='black',
)


# ---------------------------------------------------------------------------
# Calculate horizontal column boundaries
# ---------------------------------------------------------------------------

column_left_edges = [TABLE_LEFT]

running_x = TABLE_LEFT

for width_fraction in col_widths[:-1]:
    running_x += TABLE_WIDTH * width_fraction
    column_left_edges.append(running_x)

column_display_widths = [
    TABLE_WIDTH * width_fraction
    for width_fraction in col_widths
]


# ---------------------------------------------------------------------------
# Draw header row
# ---------------------------------------------------------------------------

header_bottom = table_top - header_height

for ci, label in enumerate(col_labels):

    cell_left = column_left_edges[ci]
    cell_width = column_display_widths[ci]

    header_rectangle = Rectangle(
        (cell_left, header_bottom),
        cell_width,
        header_height,
        facecolor=HEADER_BACKGROUND,
        edgecolor=GRID_COLOR,
        linewidth=GRID_LINEWIDTH,
    )

    ax_table.add_patch(header_rectangle)

    ax_table.text(
        cell_left + cell_width / 2,
        header_bottom + header_height / 2,
        label,
        fontproperties=header_font,
        color='white',
        ha='center',
        va='center',
    )


# ---------------------------------------------------------------------------
# Draw body rows
# ---------------------------------------------------------------------------

current_top = header_bottom

for ri, row in enumerate(wrapped_rows):

    row_height = row_heights[ri]
    row_bottom = current_top - row_height
    cat_color = row['color']

    for ci, cell_text in enumerate(row['text']):

        cell_left = column_left_edges[ci]
        cell_width = column_display_widths[ci]

        if ci == 0:
            facecolor = cat_color
            text_color = (
                '#222222'
                if cat_color in ('#FFDC00', '#F39C12')
                else 'white'
            )
            font_properties = category_font
        else:
            facecolor = BODY_BACKGROUND
            text_color = '#222222'
            font_properties = body_font

        cell_rectangle = Rectangle(
            (cell_left, row_bottom),
            cell_width,
            row_height,
            facecolor=facecolor,
            edgecolor=GRID_COLOR,
            linewidth=GRID_LINEWIDTH,
        )
        ax_table.add_patch(cell_rectangle)

        ax_table.text(
            cell_left + horizontal_padding_fraction,
            row_bottom + row_height / 2,
            cell_text,
            fontproperties=font_properties,
            color=text_color,
            ha='left',
            va='center',
            multialignment='left',
            linespacing=BODY_LINE_SPACING,
            clip_on=True,
        )

    current_top = row_bottom


# ---------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------

output_file = 'panel_A_locus_criteria.png'

plt.savefig(
    output_file,
    dpi=300,
    facecolor='white',
    bbox_inches='tight',
    pad_inches=0.025,
)

plt.show()

print(f'\nSaved: {output_file}')

## Panel B — Distribution of locus coverage categories

Pie chart showing the proportion of annotated paralog loci assigned to
each coverage category. Each locus belongs to exactly one category.


In [ ]:
# ============================================================================
# Panel B — Locus coverage category distribution
# Clean publication-style pie chart with compact external legend
# ============================================================================

import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# ---------------------------------------------------------------------------
# Publication font sizes
# ---------------------------------------------------------------------------

TITLE_FS = 18
PERCENT_FS = 13.5
LEGEND_FS = 12


# ---------------------------------------------------------------------------
# Count and order categories
# ---------------------------------------------------------------------------

locus_counts = (
    paralog_loci['locus_coverage_category']
    .value_counts()
    .reindex([
        category
        for category in LOCUS_ORDER
        if category in paralog_loci[
            'locus_coverage_category'
        ].unique()
    ])
    .fillna(0)
    .astype(int)
)

total_n = int(locus_counts.sum())

categories = list(locus_counts.index)
values = locus_counts.values
colors_pie = [LOCUS_COLORS[category] for category in categories]


# ---------------------------------------------------------------------------
# Percentage formatter
# ---------------------------------------------------------------------------

def percentage_label(pct):
    """
    Show percentages only for slices comprising at least 2% of all loci.

    Smaller categories remain visible as colored wedges and are reported
    in the external legend.
    """
    return f'{pct:.1f}%' if pct >= 2.0 else ''


# ---------------------------------------------------------------------------
# Create figure
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11.2, 6.8))

wedges, _, autotexts = ax.pie(
    values,
    labels=None,
    colors=colors_pie,

    # Places the smaller category cluster on the right side
    startangle=0,
    counterclock=True,

    radius=1.0,

    autopct=percentage_label,
    pctdistance=0.65,

    wedgeprops={
        'edgecolor': 'white',
        'linewidth': 1.6,
    },

    textprops={
        'fontsize': PERCENT_FS,
        'color': '#222222',
    },
)


# ---------------------------------------------------------------------------
# Format percentages inside wedges
# ---------------------------------------------------------------------------

for text in autotexts:
    text.set_fontsize(PERCENT_FS)
    text.set_fontweight('bold')
    text.set_color('#222222')


# ---------------------------------------------------------------------------
# Build compact legend labels
# ---------------------------------------------------------------------------

legend_handles = []
legend_labels = []

for category, count in locus_counts.items():

    percentage = 100 * count / total_n
    display_name = LOCUS_DISPLAY.get(category, category).upper()

    legend_handles.append(
        Patch(
            facecolor=LOCUS_COLORS[category],
            edgecolor='none',
        )
    )

    legend_labels.append(
        f'{display_name}\n'
        f'n = {count:,}  ({percentage:.1f}%)'
    )


# ---------------------------------------------------------------------------
# Draw external legend
# ---------------------------------------------------------------------------

legend = ax.legend(
    handles=legend_handles,
    labels=legend_labels,

    loc='center left',
    bbox_to_anchor=(1.01, 0.50),

    frameon=False,

    fontsize=LEGEND_FS,
    labelspacing=1.10,

    handlelength=1.25,
    handleheight=1.25,
    handletextpad=0.75,

    borderaxespad=0,
)

# Left-align multiline legend entries
for text in legend.get_texts():
    text.set_multialignment('left')


# ---------------------------------------------------------------------------
# Title and layout
# ---------------------------------------------------------------------------

ax.set_title(
    f'Distribution of category within {total_n:,} paralog loci',
    fontsize=TITLE_FS,
    fontweight='bold',
    x=0.7,   # Horizontal centering
    # Controls the vertical placement of the title
    y=1.05,
    pad=0,
)

ax.set_aspect('equal')

# Tight asymmetric limits reduce whitespace above the pie and shift the
# visual center slightly upward toward the title.
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.15, 1.03)

# Leave room for the external legend
fig.subplots_adjust(
    left=0.04,
    right=0.72,
    top=0.94,
    bottom=0.04,
)


# ---------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------

output_file = 'panel_B_locus_pie.png'

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.08,
    facecolor='white',
)

plt.show()

print(f'Saved: {output_file}')


# ---------------------------------------------------------------------------
# Print counts
# ---------------------------------------------------------------------------

print('\nLocus coverage categories (paralogs):')
print(f'{"Category":45s} {"N":>6s} {"%":>7s}')
print('-' * 61)

for category, count in locus_counts.items():

    percentage = 100 * count / total_n

    print(
        f'{LOCUS_DISPLAY.get(category, category):45s} '
        f'{count:6d} '
        f'{percentage:6.1f}%'
    )

## Panel C — Coverage category by paralog type

Stacked bar chart showing the distribution of locus coverage categories
across paralog classes (l: Liftoff-only, dl: de novo + Liftoff,
rl: reference-based Liftoff).


In [ ]:
# ============================================================================
# Panel C — Locus coverage category by paralog type
# Publication-style 100% stacked bar chart with larger, legible fonts
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# Publication font sizes
# ---------------------------------------------------------------------------

AXIS_LABEL_FS = 15
TICK_LABEL_FS = 13
PERCENT_LABEL_FS = 12
TITLE_FS = 17


# ---------------------------------------------------------------------------
# Keep paralog loci with an assigned paralog type
# ---------------------------------------------------------------------------

paralog_only = paralog_loci[
    paralog_loci['paralog_type'].notna()
    & (paralog_loci['paralog_type'] != '')
].copy()


# ---------------------------------------------------------------------------
# Absolute and percentage cross-tabulations
# ---------------------------------------------------------------------------

ct_abs = pd.crosstab(
    paralog_only['locus_coverage_category'],
    paralog_only['paralog_type'],
)

ct_abs = (
    ct_abs
    .reindex(
        index=[
            category
            for category in LOCUS_ORDER
            if category in ct_abs.index
        ],
        columns=PARALOG_TYPE_ORDER,
    )
    .fillna(0)
    .astype(int)
)

ct_pct = ct_abs.div(
    ct_abs.sum(axis=0),
    axis=1,
) * 100


# ---------------------------------------------------------------------------
# Create figure
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 6.5))

x_positions = np.arange(len(PARALOG_TYPE_ORDER))
bottom = np.zeros(len(PARALOG_TYPE_ORDER))


# ---------------------------------------------------------------------------
# Draw stacked bars
# ---------------------------------------------------------------------------

for category in ct_pct.index:

    values = ct_pct.loc[category].values

    ax.bar(
        x_positions,
        values,
        bottom=bottom,

        color=LOCUS_COLORS[category],
        label=LOCUS_DISPLAY.get(category, category),

        width=0.60,

        edgecolor='white',
        linewidth=1.0,
    )

    # Add percentage labels only to sufficiently large segments
    for i, value in enumerate(values):

        if value >= 5:

            ax.text(
                x_positions[i],
                bottom[i] + value / 2,
                f'{value:.0f}%',

                ha='center',
                va='center',

                fontsize=PERCENT_LABEL_FS,
                fontweight='bold',
                color='#111111',
            )

    bottom += values


# ---------------------------------------------------------------------------
# X-axis labels
# ---------------------------------------------------------------------------

x_labels = [
    f'{paralog_type}\n'
    f'(n = {int(ct_abs[paralog_type].sum()):,})'
    for paralog_type in PARALOG_TYPE_ORDER
]

ax.set_xticks(x_positions)

ax.set_xticklabels(
    x_labels,
    fontsize=TICK_LABEL_FS,
    linespacing=1.25,
)


# ---------------------------------------------------------------------------
# Y-axis formatting
# ---------------------------------------------------------------------------

ax.set_ylabel(
    'Percentage of paralog loci',
    fontsize=AXIS_LABEL_FS,
    labelpad=10,
)

ax.set_ylim(0, 105)

ax.set_yticks(
    np.arange(0, 101, 20)
)

ax.tick_params(
    axis='y',
    labelsize=TICK_LABEL_FS,
    width=1.0,
    length=5,
    pad=7,
)

ax.tick_params(
    axis='x',
    labelsize=TICK_LABEL_FS,
    width=1.0,
    length=5,
    pad=8,
)


# ---------------------------------------------------------------------------
# Optional title
# ---------------------------------------------------------------------------

# ax.set_title(
#     'Locus coverage categories by paralog type',
#     fontsize=TITLE_FS,
#     fontweight='normal',
#     pad=10,
# )


# ---------------------------------------------------------------------------
# Optional legend
# ---------------------------------------------------------------------------

# legend = ax.legend(
#     loc='upper left',
#     bbox_to_anchor=(1.01, 1.00),
#
#     frameon=False,
#
#     fontsize=11.5,
#     title='Locus category',
#     title_fontsize=12.5,
#
#     labelspacing=0.9,
#     handlelength=1.2,
#     handleheight=1.2,
#     borderaxespad=0,
# )
#
# for text in legend.get_texts():
#     text.set_multialignment('left')


# ---------------------------------------------------------------------------
# Clean publication-style axes
# ---------------------------------------------------------------------------

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_linewidth(1.1)
ax.spines['bottom'].set_linewidth(1.1)

ax.grid(False)


# ---------------------------------------------------------------------------
# Layout and export
# ---------------------------------------------------------------------------

fig.subplots_adjust(
    left=0.14,
    right=0.98,
    top=0.97,
    bottom=0.16,
)

output_file = 'panel_C_locus_by_paralog_type.png'

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.08,
    facecolor='white',
)

plt.show()

print(f'Saved: {output_file}')


# ---------------------------------------------------------------------------
# Print cross-tabulation
# ---------------------------------------------------------------------------

ct_print = ct_abs.copy()

ct_print['Total'] = ct_print.sum(axis=1)
ct_print.loc['Total'] = ct_print.sum(axis=0)

print('\n=== Locus coverage category × paralog type ===')
print(ct_print.to_string())

## Panels D–I — Representative locus examples

One representative paralog locus for each coverage category. Each panel
shows ONLY the paralog locus (not the parent). Coverage values are shown
for four regions: upstream flank, gene body, downstream flank, and combined
gene ±25 kb. BUSCO-derived thresholds are overlaid as horizontal bands.


In [ ]:
# ============================================================================
# Panels D–I — Combined representative locus examples
# One example per locus-coverage category with one shared legend
# ============================================================================

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# ---------------------------------------------------------------------------
# Select one representative paralog per category
# ---------------------------------------------------------------------------

np.random.seed(42)

locus_examples = {}

for cat in LOCUS_ORDER:

    subset = paralog_loci[
        paralog_loci['locus_coverage_category'] == cat
    ]

    if len(subset) == 0:
        continue

    # UNEVALUABLE (inconclusive) example: restrict to the definitional
    # "both flanks uninformative" sub-type (not the intermediate-depth
    # catch-all), so the panel matches the figure text.
    if cat == 'inconclusive':
        uninformative = subset[
            (subset['upstream_informative'] == False) &
            (subset['downstream_informative'] == False)
        ]
        if len(uninformative) > 0:
            subset = uninformative

    # Prefer dl-type paralogs because they are the most common type
    dl_subset = subset[
        subset['paralog_type'] == 'dl'
    ]

    if len(dl_subset) > 0:
        pick = dl_subset.sample(
            1,
            random_state=42,
        ).iloc[0]
    else:
        pick = subset.sample(
            1,
            random_state=42,
        ).iloc[0]

    locus_examples[cat] = pick

    print(
        f'{cat:45s} → '
        f'{pick["gene_name"]:25s}  '
        f'type={pick.get("paralog_type", "?")}'
    )




# ---------------------------------------------------------------------------
# Region labels
# ---------------------------------------------------------------------------

region_labels = [
    'Upstream\nflank',
    'Gene\nbody',
    'Downstream\nflank',
    'Combined\n±25 kb',
]


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def safe_float(value):
    """
    Convert a value to float where possible.
    Returns NaN for missing or nonnumeric values.
    """

    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def is_true(value):
    """
    Safely interpret diagnostic flags while treating missing values as False.
    """

    if value is None:
        return False

    try:
        if np.isnan(value):
            return False
    except (TypeError, ValueError):
        pass

    return bool(value)


def extract_coverage_values(row):
    """
    Return unique and permissive coverage values in plotting order.
    """

    unique_values = [
        safe_float(row.get('upstream_unique_chr_norm', np.nan)),
        safe_float(row.get('gene_body_unique_chr_norm', np.nan)),
        safe_float(row.get('downstream_unique_chr_norm', np.nan)),
        safe_float(row.get('combined_unique_chr_norm', np.nan)),
    ]

    permissive_values = [
        safe_float(row.get('upstream_permissive_chr_norm', np.nan)),
        safe_float(row.get('gene_body_permissive_chr_norm', np.nan)),
        safe_float(row.get('downstream_permissive_chr_norm', np.nan)),
        safe_float(row.get('combined_permissive_chr_norm', np.nan)),
    ]

    return unique_values, permissive_values


def build_flag_text(row):
    """
    Build the compact diagnostic summary shown within each panel.
    """

    flags = []

    both_info = is_true(
        row.get('both_flanks_informative', False)
    )

    sym_half = is_true(
        row.get('symmetric_half_depth', False)
    )

    map_def = is_true(
        row.get('has_mapping_deficit', False)
    )

    comb_half = is_true(
        row.get('combined_is_half', False)
    )

    comb_dipl = is_true(
        row.get('combined_is_diploid', False)
    )

    gb_dipl = is_true(
        row.get('gene_body_is_diploid', False)
    )

    up_half = is_true(
        row.get('upstream_is_half', False)
    )

    dn_half = is_true(
        row.get('downstream_is_half', False)
    )

    up_dipl = is_true(
        row.get('upstream_is_diploid', False)
    )

    dn_dipl = is_true(
        row.get('downstream_is_diploid', False)
    )

    up_vlow = is_true(
        row.get('upstream_is_very_low', False)
    )

    dn_vlow = is_true(
        row.get('downstream_is_very_low', False)
    )

    if both_info:
        flags.append('flanks informative')
    else:
        flags.append('flanks not informative')

    if sym_half:
        flags.append('symmetric half-depth')

    if map_def:
        flags.append('mapping deficit')

    if comb_half:
        flags.append('combined half-depth')

    if comb_dipl:
        flags.append('combined expected')

    if gb_dipl:
        flags.append('gene body expected')

    if up_half or dn_half:

        sides = []

        if up_half:
            sides.append('up')

        if dn_half:
            sides.append('down')

        flags.append(
            f'half-depth flank: {", ".join(sides)}'
        )

    if up_dipl or dn_dipl:

        sides = []

        if up_dipl:
            sides.append('up')

        if dn_dipl:
            sides.append('down')

        flags.append(
            f'expected flank: {", ".join(sides)}'
        )

    if up_vlow or dn_vlow:

        sides = []

        if up_vlow:
            sides.append('up')

        if dn_vlow:
            sides.append('down')

        flags.append(
            f'very-low flank: {", ".join(sides)}'
        )

    return '\n'.join(flags)


# ---------------------------------------------------------------------------
# Determine category order actually available for plotting
# ---------------------------------------------------------------------------

plot_categories = [
    cat
    for cat in LOCUS_ORDER
    if cat in locus_examples
]

n_panels = len(plot_categories)

if n_panels == 0:
    raise ValueError(
        'No representative locus examples were available for plotting.'
    )


# ---------------------------------------------------------------------------
# Calculate one shared y-axis maximum
# ---------------------------------------------------------------------------

all_coverage_values = []

for cat in plot_categories:

    unique_values, permissive_values = extract_coverage_values(
        locus_examples[cat]
    )

    all_coverage_values.extend(unique_values)
    all_coverage_values.extend(permissive_values)


finite_coverage_values = [
    value
    for value in all_coverage_values
    if np.isfinite(value)
]

maximum_observed = max(
    finite_coverage_values + [EXPECTED_HI]
)

# Leave enough room for bar labels
shared_ymax = maximum_observed * 1.22 + 0.08


# ---------------------------------------------------------------------------
# Create combined multipanel figure
# ---------------------------------------------------------------------------

n_cols = 3
n_rows = math.ceil(n_panels / n_cols)

fig, axes = plt.subplots(
    nrows=n_rows,
    ncols=n_cols,
    figsize=(16, 9.5),
    sharey=True,
)

axes = np.atleast_1d(axes).ravel()

x = np.arange(
    len(region_labels)
)

bar_width = 0.31


# ---------------------------------------------------------------------------
# Draw each locus panel
# ---------------------------------------------------------------------------

for panel_index, cat in enumerate(plot_categories):

    ax = axes[panel_index]

    row = locus_examples[cat]

    gene_name = row['gene_name']
    category_display = LOCUS_DISPLAY.get(cat, cat)

    unique_values, permissive_values = extract_coverage_values(
        row
    )


    # -----------------------------------------------------------------------
    # Threshold bands
    # -----------------------------------------------------------------------

    ax.axhspan(
        EXPECTED_LO,
        EXPECTED_HI,
        color='#2ECC40',
        alpha=0.10,
        zorder=0,
    )

    ax.axhspan(
        HALF_LO,
        HALF_HI,
        color='#E74C3C',
        alpha=0.10,
        zorder=0,
    )

    ax.axhspan(
        VERY_LOW_LO,
        VERY_LOW_HI,
        color='#FF4136',
        alpha=0.10,
        zorder=0,
    )


    # -----------------------------------------------------------------------
    # Threshold boundaries
    # -----------------------------------------------------------------------

    ax.axhline(
        EXPECTED_LO,
        color='#27AE60',
        linestyle='--',
        linewidth=0.85,
        alpha=0.75,
        zorder=1,
    )

    ax.axhline(
        EXPECTED_HI,
        color='#27AE60',
        linestyle='--',
        linewidth=0.85,
        alpha=0.75,
        zorder=1,
    )

    ax.axhline(
        HALF_LO,
        color='#C0392B',
        linestyle=':',
        linewidth=0.85,
        alpha=0.75,
        zorder=1,
    )

    ax.axhline(
        HALF_HI,
        color='#C0392B',
        linestyle=':',
        linewidth=0.85,
        alpha=0.75,
        zorder=1,
    )

    ax.axhline(
        VERY_LOW_LO,
        color='#E74C3C',
        linestyle='-.',
        linewidth=0.85,
        alpha=0.75,
        zorder=1,
    )


    # -----------------------------------------------------------------------
    # Coverage bars
    # -----------------------------------------------------------------------

    bars_unique = ax.bar(
        x - bar_width / 2,
        unique_values,
        bar_width,
        color='#3498DB',
        edgecolor='white',
        linewidth=0.6,
        zorder=3,
    )

    bars_permissive = ax.bar(
        x + bar_width / 2,
        permissive_values,
        bar_width,
        color='#95A5A6',
        edgecolor='white',
        linewidth=0.6,
        zorder=3,
    )


    # -----------------------------------------------------------------------
    # Bar-value labels
    # -----------------------------------------------------------------------

    label_offset = shared_ymax * 0.018

    for i, (unique_value, permissive_value) in enumerate(
        zip(unique_values, permissive_values)
    ):

        if np.isfinite(unique_value):

            ax.text(
                i - bar_width / 2,
                unique_value + label_offset,
                f'{unique_value:.2f}',
                ha='center',
                va='bottom',
                fontsize=8,
                fontweight='bold',
                color='black',
                clip_on=False,
                zorder=5,
            )

        if np.isfinite(permissive_value):

            ax.text(
                i + bar_width / 2,
                permissive_value + label_offset,
                f'{permissive_value:.2f}',
                ha='center',
                va='bottom',
                fontsize=8,
                fontweight='bold',
                color='black',
                clip_on=False,
                zorder=5,
            )


    # -----------------------------------------------------------------------
    # Axes formatting
    # -----------------------------------------------------------------------
    
    ax.set_xticks(x)
    
    ax.set_xticklabels(
        region_labels,
        fontsize=9,
    )
    
    ax.set_ylim(
        0,
        shared_ymax,
    )
    
    # Keep y-axis ticks and labels only on the left column
    if panel_index % n_cols == 0:
    
        ax.tick_params(
            axis='y',
            which='both',
            left=True,
            right=False,
            labelleft=True,
            labelsize=9,
        )
    
        ax.set_ylabel(
            'Normalized coverage\n(× chr-class baseline)',
            fontsize=10,
            labelpad=8,
        )
    
    else:
    
        ax.tick_params(
            axis='y',
            which='both',
            left=False,
            right=False,
            labelleft=False,
        )
    
        # Hide the left spine on non-left-column panels
        ax.spines['left'].set_visible(False)
    
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.grid(
        axis='y',
        color='#D9D9D9',
        linewidth=0.5,
        alpha=0.5,
        zorder=0,
    )


    # -----------------------------------------------------------------------
    # Panel title
    # Match the formatting used in the expression example panels
    # -----------------------------------------------------------------------
    
    ax.set_title(
        f'{category_display.upper()}\n'
        f'Locus: {gene_name}',
        fontsize=11,
        fontweight='bold',
        color='#222222',
        pad=5,
    )
    
    # -----------------------------------------------------------------------
    # Diagnostic summary
    # -----------------------------------------------------------------------

    flag_text = build_flag_text(row)

    ax.text(
        0.02,
        0.96,
        flag_text,
        transform=ax.transAxes,
        fontsize=7.3,
        color='#333333',
        ha='left',
        va='top',
        linespacing=1.15,
        bbox={
            'boxstyle': 'round,pad=0.35',
            'facecolor': 'white',
            'edgecolor': LOCUS_COLORS.get(cat, '#777777'),
            'linewidth': 1.0,
            'alpha': 0.90,
        },
        zorder=6,
    )


# ---------------------------------------------------------------------------
# Remove unused axes
# ---------------------------------------------------------------------------

for unused_index in range(n_panels, len(axes)):
    fig.delaxes(
        axes[unused_index]
    )


# ---------------------------------------------------------------------------
# One shared legend
# ---------------------------------------------------------------------------

legend_elements = [
    mpatches.Patch(
        facecolor='#3498DB',
        edgecolor='white',
        label='Unique coverage (MAPQ ≥ 20)',
    ),
    mpatches.Patch(
        facecolor='#95A5A6',
        edgecolor='white',
        label='Permissive coverage (MAPQ ≥ 0)',
    ),
    mpatches.Patch(
        facecolor='#2ECC40',
        alpha=0.20,
        label=(
            f'Expected depth '
            f'[{EXPECTED_LO}–{EXPECTED_HI}]'
        ),
    ),
    mpatches.Patch(
        facecolor='#E74C3C',
        alpha=0.20,
        label=(
            f'Half-depth '
            f'[{HALF_LO}–{HALF_HI}]'
        ),
    ),
    mpatches.Patch(
        facecolor='#FF4136',
        alpha=0.20,
        label=(
            f'Very-low depth '
            f'[{VERY_LOW_LO}–{VERY_LOW_HI})'
        ),
    ),
]

shared_legend = fig.legend(
    handles=legend_elements,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.025),
    ncol=5,
    frameon=False,
    fontsize=9.5,
    handlelength=1.5,
    columnspacing=1.5,
)

for legend_text in shared_legend.get_texts():
    legend_text.set_color('#222222')


# ---------------------------------------------------------------------------
# Overall title and figure spacing
# ---------------------------------------------------------------------------

fig.subplots_adjust(
    left=0.075,
    right=0.985,
    top=0.91,
    bottom=0.14,
    wspace=0.12,
    hspace=0.34,
)


# ---------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------

output_file = 'panels_D_to_I_locus_examples_combined.png'

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.08,
    facecolor='white',
)

plt.show()

print(f'Saved: {output_file}')

## Panel J — Pair-level interpretation framework

After each locus is independently classified, parent–paralog pairs are
assigned an interpretation by combining the two locus classifications.
The sum check (parent + paralog combined depth ≈ 1.0) is used as a
secondary validator: two half-depth loci that sum to the expected
single-copy baseline are consistent with a single diploid locus
represented as two haplotypic sequences.

This panel explains how two independently classified loci are combined
into biological interpretations. It is intentionally an interpretation
framework rather than another coverage classifier.


In [ ]:
# ============================================================================
# Panel J — Pair-level interpretation framework
# Publication-ready manually drawn table with category counts
# All category names use white text
# ============================================================================

import re
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from matplotlib.font_manager import FontProperties


# ---------------------------------------------------------------------------
# Count source
# ---------------------------------------------------------------------------

# Change these two lines only if your dataframe or column has another name.
PAIR_DATA = pairs_v4
PAIR_CATEGORY_COLUMN = 'pair_interpretation'

# Count the number of pairs assigned to each pair-level category.
raw_pair_counts = (
    PAIR_DATA[PAIR_CATEGORY_COLUMN]
    .value_counts(dropna=False)
    .to_dict()
)


# ---------------------------------------------------------------------------
# Category-name normalization for reliable count matching
# ---------------------------------------------------------------------------

def normalize_category_key(value):
    """
    Convert category names to a consistent lookup format.

    Examples:
        'BOTH LOCI AT EXPECTED DEPTH'
            -> 'both_loci_at_expected_depth'

        'both-expected-depth'
            -> 'both_expected_depth'
    """
    value = str(value).strip().lower()
    value = re.sub(r'[^a-z0-9]+', '_', value)
    return value.strip('_')


normalized_pair_counts = {
    normalize_category_key(category): int(count)
    for category, count in raw_pair_counts.items()
}


def get_pair_count(category_key, display_name):
    """
    Retrieve the pair count using either the internal key or display name.
    Returns zero if neither form is found.
    """

    candidate_keys = [
        normalize_category_key(category_key),
        normalize_category_key(display_name),
    ]

    for candidate in candidate_keys:
        if candidate in normalized_pair_counts:
            return normalized_pair_counts[candidate]

    return 0


# ---------------------------------------------------------------------------
# Pair-level classifications
# ---------------------------------------------------------------------------

# Tuple:
# (
#     internal_category_key,
#     display_name,
#     color,
#     locus_combination,
#     biological_interpretation,
# )

pair_criteria_examples = [
    (
        'both_expected_depth',
        'BOTH LOCI AT EXPECTED DEPTH',
        '#2ECC40',
        'Both loci = EXPECTED DEPTH',
        PAIR_INTERPRETATION_TEXT['both_expected_depth'],
    ),
    (
        'one_expected_one_atypical',
        'ONE EXPECTED, ONE ATYPICAL',
        '#FFDC00',
        'One locus = EXPECTED DEPTH; other = LOCALIZED LOW DEPTH, '
        'UNEVALUABLE, or MAPPING AMBIGUITY',
        PAIR_INTERPRETATION_TEXT['one_expected_one_atypical'],
    ),
    (
        'both_atypical',
        'BOTH ATYPICAL',
        '#F39C12',
        'At least one locus = LOCALIZED LOW DEPTH, VERY LOW '
        'COVERAGE, or MAPPING AMBIGUITY; the other may be '
        'UNEVALUABLE; not both EXPECTED DEPTH or both HALF-DEPTH',
        PAIR_INTERPRETATION_TEXT['both_atypical'],
    ),
    (
        'asymmetric_coverage',
        'ASYMMETRIC COVERAGE',
        '#FF851B',
        'One locus = EXPECTED DEPTH; other = HALF-DEPTH or '
        'VERY LOW COVERAGE',
        PAIR_INTERPRETATION_TEXT['asymmetric_coverage'],
    ),
    (
        'unresolved',
        'INDETERMINATE',
        '#AAAAAA',
        'Both loci = UNEVALUABLE',
        PAIR_INTERPRETATION_TEXT['unresolved'],
    ),
    (
        'concordant_half_depth_sum_valid',
        'CONCORDANT HALF-DEPTH',
        '#E74C3C',
        'Both loci = HALF-DEPTH. Sum check ≈ 1.0 when loci are '
        'non-overlapping, in the same chromosome class, and both '
        'flanks are informative',
        PAIR_INTERPRETATION_TEXT[
            'concordant_half_depth_sum_valid'
        ],
    ),
]


# ---------------------------------------------------------------------------
# Add counts to classification records
# ---------------------------------------------------------------------------

pair_criteria_with_counts = []

for (
    category_key,
    display_name,
    category_color,
    locus_combination,
    biological_interpretation,
) in pair_criteria_examples:

    category_count = get_pair_count(
        category_key=category_key,
        display_name=display_name,
    )

    pair_criteria_with_counts.append(
        (
            category_key,
            display_name,
            category_color,
            locus_combination,
            biological_interpretation,
            category_count,
        )
    )


# ---------------------------------------------------------------------------
# Column labels and widths
# ---------------------------------------------------------------------------

col_labels_pair = [
    'Pair classification',
    'Locus combination',
    'Biological interpretation',
    'Count (n)',
]

# Must sum to exactly 1.0
col_widths_pair = [
    0.20,   # Pair classification
    0.32,   # Locus combination
    0.41,   # Biological interpretation
    0.07,   # Count
]

assert abs(sum(col_widths_pair) - 1.0) < 1e-8


# ---------------------------------------------------------------------------
# Figure and typography settings
# ---------------------------------------------------------------------------

FIGURE_WIDTH_IN = 18.5

TITLE_FONTSIZE = 20
HEADER_FONTSIZE = 15
CATEGORY_FONTSIZE = 13
BODY_FONTSIZE = 12
COUNT_FONTSIZE = 13

BODY_LINE_SPACING = 1.04

# Internal cell padding in points
HORIZONTAL_PADDING_PT = 7.0
VERTICAL_PADDING_PT = 5.0

# Header and figure spacing in points
HEADER_HEIGHT_PT = 50.0

TOP_MARGIN_PT = 7.0
TITLE_HEIGHT_PT = 27.0
TITLE_TABLE_GAP_PT = 7.0
BOTTOM_MARGIN_PT = 7.0

# Table horizontal placement
TABLE_LEFT = 0.02
TABLE_WIDTH = 0.96

# Table appearance
HEADER_BACKGROUND = '#333333'
BODY_BACKGROUND = '#FAFAFA'

GRID_COLOR = '#666666'
GRID_LINEWIDTH = 0.70


# ---------------------------------------------------------------------------
# Font definitions
# ---------------------------------------------------------------------------

body_font = FontProperties(
    family='sans-serif',
    size=BODY_FONTSIZE,
    weight='normal',
)

category_font = FontProperties(
    family='sans-serif',
    size=CATEGORY_FONTSIZE,
    weight='bold',
)

count_font = FontProperties(
    family='sans-serif',
    size=COUNT_FONTSIZE,
    weight='bold',
)

header_font = FontProperties(
    family='sans-serif',
    size=HEADER_FONTSIZE,
    weight='bold',
)

title_font = FontProperties(
    family='sans-serif',
    size=TITLE_FONTSIZE,
    weight='bold',
)


# ---------------------------------------------------------------------------
# Text helpers
# ---------------------------------------------------------------------------

def normalize_text(text):
    """Remove manual line breaks and collapse repeated whitespace."""
    return re.sub(r'\s+', ' ', str(text)).strip()


def rendered_text_width(renderer, text, font_properties):
    """Return rendered text width in pixels."""

    width, _, _ = renderer.get_text_width_height_descent(
        text,
        font_properties,
        ismath=False,
    )

    return width


def wrap_to_rendered_width(
    text,
    max_width_px,
    renderer,
    font_properties,
):
    """
    Wrap text according to its actual rendered width rather than an
    approximate character count.
    """

    text = normalize_text(text)

    if not text:
        return ''

    words = text.split()
    lines = []
    current_line = words[0]

    for word in words[1:]:

        candidate_line = f'{current_line} {word}'

        candidate_width = rendered_text_width(
            renderer,
            candidate_line,
            font_properties,
        )

        if candidate_width <= max_width_px:
            current_line = candidate_line
        else:
            lines.append(current_line)
            current_line = word

    lines.append(current_line)

    return '\n'.join(lines)


# ---------------------------------------------------------------------------
# Print verification
# ---------------------------------------------------------------------------

print('=' * 110)
print('PANEL J — PAIR INTERPRETATION FRAMEWORK')
print('=' * 110)

for i, (
    category_key,
    interp_name,
    interp_color,
    locus_combo,
    bio_interp,
    category_count,
) in enumerate(pair_criteria_with_counts, start=1):

    print(f'\n{"─" * 110}')
    print(f'{i}. {normalize_text(interp_name)}')
    print(
        f'   Locus combination:          '
        f'{normalize_text(locus_combo)}'
    )
    print(
        f'   Biological interpretation:  '
        f'{normalize_text(bio_interp)}'
    )
    print(f'   Count:                      {category_count:,}')


# ---------------------------------------------------------------------------
# Create temporary figure for measuring rendered text
# ---------------------------------------------------------------------------

fig = plt.figure(
    figsize=(FIGURE_WIDTH_IN, 8.0),
    facecolor='white',
)

ax_table = fig.add_axes([0, 0, 1, 1])

ax_table.set_xlim(0, 1)
ax_table.set_ylim(0, 1)
ax_table.axis('off')

fig.canvas.draw()
renderer = fig.canvas.get_renderer()

figure_width_px = FIGURE_WIDTH_IN * fig.dpi
table_width_px = figure_width_px * TABLE_WIDTH

horizontal_padding_px = (
    HORIZONTAL_PADDING_PT
    / 72.0
    * fig.dpi
)


# ---------------------------------------------------------------------------
# Wrap table content according to actual column widths
# ---------------------------------------------------------------------------

wrapped_rows = []
row_line_counts = []

for (
    category_key,
    interp_name,
    interp_color,
    locus_combo,
    bio_interp,
    category_count,
) in pair_criteria_with_counts:

    raw_values = [
        interp_name,
        locus_combo,
        bio_interp,
        f'{category_count:,}',
    ]

    wrapped_values = []

    for ci, value in enumerate(raw_values):

        cell_width_px = (
            table_width_px
            * col_widths_pair[ci]
        )

        available_width_px = (
            cell_width_px
            - 2 * horizontal_padding_px
            - 6
        )

        if ci == 0:
            selected_font = category_font
        elif ci == 3:
            selected_font = count_font
        else:
            selected_font = body_font

        wrapped_value = wrap_to_rendered_width(
            text=value,
            max_width_px=available_width_px,
            renderer=renderer,
            font_properties=selected_font,
        )

        wrapped_values.append(wrapped_value)

    wrapped_rows.append({
        'text': wrapped_values,
        'color': interp_color,
        'count': category_count,
    })

    # The count column is one line and therefore will not normally
    # determine row height.
    row_line_counts.append(
        max(
            text.count('\n') + 1
            for text in wrapped_values
        )
    )


# ---------------------------------------------------------------------------
# Calculate row heights
# ---------------------------------------------------------------------------

body_line_height_pt = BODY_FONTSIZE * 1.12

row_heights_pt = []

for line_count in row_line_counts:

    row_height_pt = (
        line_count * body_line_height_pt
        + 2 * VERTICAL_PADDING_PT
    )

    row_heights_pt.append(row_height_pt)


table_height_pt = (
    HEADER_HEIGHT_PT
    + sum(row_heights_pt)
)

figure_height_pt = (
    TOP_MARGIN_PT
    + TITLE_HEIGHT_PT
    + TITLE_TABLE_GAP_PT
    + table_height_pt
    + BOTTOM_MARGIN_PT
)

figure_height_in = figure_height_pt / 72.0

fig.set_size_inches(
    FIGURE_WIDTH_IN,
    figure_height_in,
    forward=True,
)

fig.canvas.draw()


# ---------------------------------------------------------------------------
# Convert point dimensions to axes fractions
# ---------------------------------------------------------------------------

def points_to_y_fraction(points):
    return points / figure_height_pt


title_y = (
    1.0
    - points_to_y_fraction(TOP_MARGIN_PT)
)

table_top = (
    1.0
    - points_to_y_fraction(
        TOP_MARGIN_PT
        + TITLE_HEIGHT_PT
        + TITLE_TABLE_GAP_PT
    )
)

header_height = points_to_y_fraction(
    HEADER_HEIGHT_PT
)

row_heights = [
    points_to_y_fraction(height)
    for height in row_heights_pt
]

horizontal_padding_fraction = (
    HORIZONTAL_PADDING_PT
    / 72.0
    / FIGURE_WIDTH_IN
)


# ---------------------------------------------------------------------------
# Draw title
# ---------------------------------------------------------------------------

ax_table.text(
    0.5,
    title_y,
    'Pair-level integration of locus coverage classifications',
    fontproperties=title_font,
    color='black',
    ha='center',
    va='top',
)


# ---------------------------------------------------------------------------
# Calculate horizontal column boundaries
# ---------------------------------------------------------------------------

column_left_edges = [TABLE_LEFT]

running_x = TABLE_LEFT

for width_fraction in col_widths_pair[:-1]:

    running_x += (
        TABLE_WIDTH
        * width_fraction
    )

    column_left_edges.append(running_x)


column_display_widths = [
    TABLE_WIDTH * width_fraction
    for width_fraction in col_widths_pair
]


# ---------------------------------------------------------------------------
# Draw header
# ---------------------------------------------------------------------------

header_bottom = (
    table_top
    - header_height
)

for ci, label in enumerate(col_labels_pair):

    cell_left = column_left_edges[ci]
    cell_width = column_display_widths[ci]

    header_rectangle = Rectangle(
        (cell_left, header_bottom),
        cell_width,
        header_height,
        facecolor=HEADER_BACKGROUND,
        edgecolor=GRID_COLOR,
        linewidth=GRID_LINEWIDTH,
    )

    ax_table.add_patch(header_rectangle)

    ax_table.text(
        cell_left + cell_width / 2,
        header_bottom + header_height / 2,
        label,
        fontproperties=header_font,
        color='white',
        ha='center',
        va='center',
    )


# ---------------------------------------------------------------------------
# Draw body rows
# ---------------------------------------------------------------------------

current_top = header_bottom

for ri, row in enumerate(wrapped_rows):

    row_height = row_heights[ri]
    row_bottom = current_top - row_height
    interp_color = row['color']

    for ci, cell_text in enumerate(row['text']):

        cell_left = column_left_edges[ci]
        cell_width = column_display_widths[ci]

        if ci == 0:
            # Colored pair-classification cell
            facecolor = interp_color
            text_color = 'white'
            font_properties = category_font
            horizontal_alignment = 'left'
            text_x = cell_left + horizontal_padding_fraction

        elif ci == 3:
            # Count column
            facecolor = BODY_BACKGROUND
            text_color = '#222222'
            font_properties = count_font
            horizontal_alignment = 'center'
            text_x = cell_left + cell_width / 2

        else:
            # Descriptive columns
            facecolor = BODY_BACKGROUND
            text_color = '#222222'
            font_properties = body_font
            horizontal_alignment = 'left'
            text_x = cell_left + horizontal_padding_fraction

        cell_rectangle = Rectangle(
            (cell_left, row_bottom),
            cell_width,
            row_height,
            facecolor=facecolor,
            edgecolor=GRID_COLOR,
            linewidth=GRID_LINEWIDTH,
        )

        ax_table.add_patch(cell_rectangle)

        ax_table.text(
            text_x,
            row_bottom + row_height / 2,
            cell_text,
            fontproperties=font_properties,
            color=text_color,
            ha=horizontal_alignment,
            va='center',
            multialignment=horizontal_alignment,
            linespacing=BODY_LINE_SPACING,
            clip_on=True,
        )

    current_top = row_bottom


# ---------------------------------------------------------------------------
# Count validation
# ---------------------------------------------------------------------------

displayed_count_total = sum(
    row['count']
    for row in wrapped_rows
)

source_count_total = int(
    PAIR_DATA[PAIR_CATEGORY_COLUMN].notna().sum()
)

print('\nPair count validation:')
print(f'  Counts shown in table: {displayed_count_total:,}')
print(f'  Non-missing source rows: {source_count_total:,}')

if displayed_count_total != source_count_total:
    print(
        '  WARNING: The totals differ. Check whether the dataframe category '
        'names match the internal category keys used above.'
    )


# ---------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------

output_file = 'panel_J_pair_criteria.png'

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.025,
    facecolor='white',
)

plt.show()

print(f'\nSaved: {output_file}')

In [ ]:
PAIR_DATA[PAIR_DATA["pair_interpretation"]=="concordant_half_depth_sum_valid"]

## Output files

| Panel | File | Description |
|-------|------|-------------|
| A | `panel_A_locus_criteria.png` | Locus-level coverage classification criteria table |
| B | `panel_B_locus_pie.png` | Distribution of locus coverage categories (pie chart) |
| C | `panel_C_locus_by_paralog_type.png` | Coverage category by paralog type (stacked bar %) |
| D | `panel_D_normal_depth.png` | Representative example: EXPECTED DEPTH |
| E | `panel_E_consistent_half_depth.png` | Representative example: HALF-DEPTH |
| F | `panel_F_very_low_coverage.png` | Representative example: VERY LOW COVERAGE |
| G | `panel_G_localized_or_asymmetric_low_depth.png` | Representative example: LOCALIZED LOW DEPTH |
| H | `panel_H_unique_mapping_deficit.png` | Representative example: MAPPING AMBIGUITY |
| I | `panel_I_inconclusive.png` | Representative example: UNEVALUABLE (locus) |
| J | `panel_J_pair_criteria.png` | Pair-level interpretation framework table |


In [ ]:
print("\n" + "="*60)
print("ALL PANELS GENERATED")
print("="*60)
print("\nPanel files:")
import glob
for f in sorted(glob.glob('panel_*.png')):
    print(f'  {f}')
